# US Aerospace Innovation Atlas
### Where U.S. aeronautics & astronautics patents were invented, 2000–2015

This notebook extends a single-year (2012) classroom map into a 16-year geographic analysis of U.S. aerospace innovation, measured by **utility patent grants in USPC class 244 (Aeronautics & Astronautics)** broken out by metropolitan area.

**Data source.** USPTO PTMT *“Patenting In Technology Classes, Breakout By U.S. Metropolitan Area”* reports (class 244), preserved on the Internet Archive. Patents are counted by the metro area of the inventor; a patent with inventors in several metros is counted in each.

**A data-quality fix.** The original assignment matched patent records to map polygons *by metro name*. Because the 2018 boundary file renamed *Los Angeles–Long Beach–Santa Ana* to *…–Anaheim*, Los Angeles — the #1 metro with 62 patents in 2012 — silently dropped to zero. Here we join on the **CBSA code** instead, so Los Angeles is correctly ranked first.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import viz

data = viz.load()
df = data['df']
print(f"{len(df):,} rows | {df['cbsa_geoid'].nunique()} metros "
      f"| years {df['year'].min()}–{df['year'].max()} "
      f"| per-capita available: {data['has_percapita']}")
df.head()

4,848 rows | 303 metros | years 2000–2015 | per-capita available: True


,cbsa_geoid,cbsa_name,year,patent_count,class_code,class_label,category,level,national_share,population,per_100k
0,31080,"Los Angeles-Long Beach-Anaheim, CA",2000,77.0,244,Aeronautics and Astronautics,Aeronautics & Astronautics,Metropolitan Statistical Area,25.753,12838417.0,0.600
1,42660,"Seattle-Tacoma-Bellevue, WA",2000,29.0,244,Aeronautics and Astronautics,Aeronautics & Astronautics,Metropolitan Statistical Area,9.699,3449241.0,0.841
2,19100,"Dallas-Fort Worth-Arlington, TX",2000,9.0,244,Aeronautics and Astronautics,Aeronautics & Astronautics,Metropolitan Statistical Area,3.010,6392065.0,0.141
3,46060,"Tucson, AZ",2000,6.0,244,Aeronautics and Astronautics,Aeronautics & Astronautics,Metropolitan Statistical Area,2.007,981620.0,0.611
4,38060,"Phoenix-Mesa-Scottsdale, AZ",2000,9.0,244,Aeronautics and Astronautics,Aeronautics & Astronautics,Metropolitan Statistical Area,3.010,4204204.0,0.214


## 1. The national picture
Annual aerospace patent grants across all U.S. metros. The series reflects both real innovation cycles and USPTO grant-processing volume.

In [2]:
viz.national_trend(data)

## 2. The map
Patenting intensity by metro area. Switch `metric` to `'per_100k'` for a population-normalised view, or call with `animate=True` to play 2000→2015.

In [3]:
viz.choropleth(data, 2015, metric='patent_count')

## 3. Leading metros — and the Los Angeles correction
The reconstructed 2012 ranking matches the original assignment file exactly **once Los Angeles is restored to first place**.

In [4]:
top2012 = (df[df.year == 2012].nlargest(8, 'patent_count')
           [['cbsa_name', 'patent_count']].reset_index(drop=True))
top2012

,cbsa_name,patent_count
0,"Los Angeles-Long Beach-Anaheim, CA",62.0
1,"Seattle-Tacoma-Bellevue, WA",45.0
2,"Tucson, AZ",35.0
3,"Dallas-Fort Worth-Arlington, TX",18.0
4,"Hartford-West Hartford-East Hartford, CT",14.0
5,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",12.0
6,"New York-Newark-Jersey City, NY-NJ-PA",11.0
7,"St. Louis, MO-IL",11.0


In [5]:
viz.top_n_bar(data, 2012, n=15)

## 4. How concentrated is aerospace innovation?
Aerospace R&D clusters tightly around a few manufacturing and defense centers. The **top-5 share** and the **Herfindahl index (HHI)** quantify that concentration and how it shifts over time.

In [6]:
viz.concentration_trend(data)

## 5. Rise and fall of clusters
Tracking individual metros reveals the story behind the aggregate: Seattle's commercial-aircraft surge, Los Angeles' long aerospace base, and the rise of Dallas–Fort Worth and Tucson.

In [7]:
leaders = (df.groupby(['cbsa_geoid', 'cbsa_name'])['patent_count'].sum()
           .nlargest(6).reset_index())
viz.msa_trends(data, leaders['cbsa_geoid'].tolist())

## 6. Absolute vs. per-capita leaders
Big metros dominate raw counts, but normalising by population surfaces smaller, highly specialised aerospace towns (e.g. Tucson, Wichita).

In [8]:
if data['has_percapita']:
    yr = 2015
    s = df[df.year == yr]
    absolute = s.nlargest(8, 'patent_count')[['cbsa_name', 'patent_count']]
    percap = (s[s.population > 250_000]
              .nlargest(8, 'per_100k')[['cbsa_name', 'per_100k']])
    display(absolute.reset_index(drop=True))
    display(percap.reset_index(drop=True))
else:
    print('Run src/enrich.py to add per-capita metrics.')

,cbsa_name,patent_count
0,"Seattle-Tacoma-Bellevue, WA",92.0
1,"Los Angeles-Long Beach-Anaheim, CA",58.0
2,"Dallas-Fort Worth-Arlington, TX",46.0
3,"San Jose-Sunnyvale-Santa Clara, CA",24.0
4,"San Francisco-Oakland-Hayward, CA",23.0
5,"Tucson, AZ",16.0
6,"Salt Lake City, UT",16.0
7,"St. Louis, MO-IL",13.0


,cbsa_name,per_100k
0,"Seattle-Tacoma-Bellevue, WA",2.460
1,"Bremerton-Silverdale, WA",2.315
2,"Tucson, AZ",1.586
3,"Salt Lake City, UT",1.373
4,"Huntsville, AL",1.350
5,"San Jose-Sunnyvale-Santa Clara, CA",1.215
6,"Wichita, KS",1.101
7,"Santa Cruz-Watsonville, CA",1.097


## 7. Deeper analysis
Beyond levels and shares: how rankings move, which metros grow or fade, and *why* — decomposed with a shift-share model across the available technology classes (aeronautics 244, propulsion 060, avionics/navigation 701).

In [9]:
import analysis as A
allcls = data['allcls']
A.rank_bump(df, 10)

In [10]:
A.growth_bar(df, df.year.min(), df.year.max())

In [11]:
A.shift_share_fig(allcls, df.year.min(), df.year.max())

## 8. Aviation vs. Space *(optional — Google Patents / BigQuery)*
USPC 244 combines aeronautics and astronautics. Run `python src/bigquery_aerospace.py` (Google account) to split them and add assignee companies; the cell below renders the result if present.

In [12]:
import config as C
import pandas as pd, plotly.express as px
p = C.PROCESSED / 'aviation_space_national.parquet'
if p.exists():
    nat = pd.read_parquet(p)
    fig = px.area(nat, x='year', y='patents', color='category',
                  color_discrete_map={'Aviation':'#fe9929','Space':'#1f77b4'},
                  title='US aerospace patents: Aviation vs Space (CPC B64)')
    fig.show()
else:
    print('Run src/bigquery_aerospace.py to generate the Aviation/Space split.')

Run src/bigquery_aerospace.py to generate the Aviation/Space split.


## Conclusion
U.S. aerospace patenting is highly clustered in a handful of metros tied to aircraft manufacturing, defense contracting, and federal R&D — Los Angeles, Seattle, Dallas–Fort Worth, Tucson, Hartford and Wichita. Over 2000–2015 the leading centers persist, while Seattle and Dallas–Fort Worth gain share.

**Limitations.** Counts are USPC-244 utility *grants* (not applications), by inventor metro, 2000–2015; pre-2010 per-capita uses 2010 population. About 12% of small micropolitan areas in the source predate the 2018 boundary file and are omitted from the map (negligible aerospace activity).

**Going further.** With a free USPTO API key, `src/api_client.py` adds CPC B64 data (1976–2024), an aviation-vs-space split, and assignee companies.

*Sources: USPTO PTMT (via Internet Archive); U.S. Census Bureau metro population estimates; 2018 Cartographic Boundary CBSA shapefile.*